Load database


Matrix registry explanation
- object_mean;
- object_median;
- all_pixels;
- balanced_pixels.


Build example matrices
- une matrice object_mean;
- une matrice object_median;
- une matrice balanced_pixels;
- une matrice all_pixels.


Compare matrix shapes
- nombre d’observations ;
- nombre de bandes ;
- nombre d’objets ;
- nombre de pixels.


Balanced pixels strategies
- random;
- center;
- éventuellement core_random si tu l’ajoutes plus tard.


Preprocessing configs
- raw;
- absorbance;
- snv;
- msc;
- sg_smooth;
- sg_d1;
- sg_d2;
- absorbance_snv;
- absorbance_sg_smooth;
- absorbance_msc; 
- absorbance_sg_d1;
- absorbance_sg_d2;
- absorbance_snv_sg_smooth;
- absorbance_snv_sg_d1;
- absorbance_snv_sg_d2.


Visual comparison of preprocessing
- spectres bruts ;
- absorbance ;
- SNV ;
- MSC ;
- SG smoothing ;
- SG derivative ;
- combinaisons.


Conclusion
- quels preprocessings sont plausibles chimiométriquement ;
- lesquels garder pour PCA ;
- lesquels garder pour SIMCA.


Sorties attendues
- results/preprocessing/matrix_shapes.csv
- results/preprocessing/preprocessing_visual_summary.csv

# 02 — Matrices and spectral preprocessing

This notebook documents and checks how the NIR UCO object database is converted into numerical matrices for modelling.

Main objectives:

1. Load the validated NIR UCO database.
2. Build modelling matrices from `object_db`.
3. Compare matrix representations:
   - `object_mean`
   - `object_median`
   - `balanced_pixels`
   - `all_pixels`
4. Compare balanced pixel sampling strategies:
   - `random`
   - `center`
5. Apply and visualize spectral preprocessing methods:
   - raw
   - absorbance
   - SNV
   - MSC
   - Savitzky-Golay smoothing
   - Savitzky-Golay derivative
   - combined preprocessing chains
6. Save matrix and preprocessing summaries for the next PCA/SIMCA notebooks.

This notebook does not fit PCA or SIMCA models yet.

In [2]:
from __future__ import annotations

import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 250)

# ---------------------------------------------------------------------
# Project root detection
# ---------------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Launch the notebook from the project "
        "root or from the notebooks/ folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [3]:
from src.io.database_h5 import load_nir_uco_h5

from src.matrices.matrix_registry import (
    build_matrix,
    available_matrix_methods,
    get_matrix_spec,
)

from src.spectra.preprocessing import SpectralPreprocessor
from src.spectra.preprocessing_configs import (
    DEFAULT_PREPROCESSING_CONFIGS,
    SIMCA_SEARCH_PREPROCESSING_CONFIGS,
    normalize_preprocessing_configs,
)
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.visualization.plot_spectra import plot_spectra
from src.visualization.plot_generic import (
    plot_bar_values,
    plot_counts_by_group,
)

from src.utils import save_parquet

%load_ext autoreload
%autoreload 2

In [4]:
# ---------------------------------------------------------------------
# Input database
# ---------------------------------------------------------------------
DB_H5_PATH = (
    PROJECT_ROOT
    / "HSI Data"
    / "processed"
    / "nir_uco_database.h5"
)
# WAVELENGHTS SELECTION
USE_SPECTRAL_RANGE = True
SPECTRAL_MIN_NM = 1225.0
SPECTRAL_MAX_NM = 1675.0
SPECTRAL_RANGE_TAG = "1225_1675"

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
#RESULTS_DIR = PROJECT_ROOT / "results" / "matrices_preprocessing"
#RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PROJECT_ROOT / "results" / f"matrices_preprocessing_{SPECTRAL_RANGE_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WAVELENGTH_SELECTION_PATH = RESULTS_DIR / "wavelength_selection.parquet"

MATRIX_SUMMARY_PATH = RESULTS_DIR / "matrix_summary.parquet"
PREPROCESSING_SUMMARY_PATH = RESULTS_DIR / "preprocessing_summary.parquet"
MATRIX_CONFIG_JSON = RESULTS_DIR / "matrices_preprocessing_config.json"

# ---------------------------------------------------------------------
# Matrix construction parameters
# ---------------------------------------------------------------------
RANDOM_STATE = 42
M_BALANCED_PIXELS = 40
REPLACE_BALANCED_PIXELS = False

BALANCED_PIXEL_STRATEGIES = [
    "random",
    "center",
]

MATRIX_METHODS_TO_CHECK = [
    "object_mean",
    "object_median",
    "balanced_pixels",
    "all_pixels",
]

# ---------------------------------------------------------------------
# Filters for this notebook
# ---------------------------------------------------------------------
# Use pure objects for preprocessing/matrix inspection.
PURE_FILTERS = {
    "sample_kind": ["pure"],
}

PURE_ALMOND_PEANUT_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": ["almond", "peanut"],
}

PURE_BATCH_12_FILTERS = {
    "sample_kind": ["pure"],
    "batch": [1, 2],
}

# ---------------------------------------------------------------------
# Preprocessing parameters
# ---------------------------------------------------------------------
SG_WINDOW_LENGTH = 11
SG_POLYORDER = 2

# Keep a compact list for visual comparison.
PREPROCESSING_CONFIGS_TO_COMPARE = {
    "raw": ("raw",),
    "absorbance": ("absorbance",),
    "snv": ("snv",),
    "msc": ("msc",),
    "sg_smooth": ("sg_smooth",),
    "sg_d1": ("sg_d1",),
    "sg_d2": ("sg_d2",),
    "snv_sg_smooth": ("snv", "sg_smooth"),
    "snv_sg_d1": ("snv", "sg_d1"),
    "snv_sg_d2": ("snv", "sg_d2"),
    "absorbance_snv": ("absorbance", "snv"),
    "absorbance_sg_smooth": ("absorbance", "sg_smooth"),
    "absorbance_sg_d1": ("absorbance", "sg_d1"),
    "absorbance_sg_d2": ("absorbance", "sg_d2"),
    "absorbance_snv_sg_smooth": ("absorbance", "snv", "sg_smooth"),
    "absorbance_snv_sg_d1": ("absorbance", "snv", "sg_d1"),
    "absorbance_snv_sg_d2": ("absorbance", "snv", "sg_d2"),
}

# Number of spectra shown in detailed raw/preprocessed plots
MAX_SPECTRA_TO_PLOT = 80

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\matrices_preprocessing_1225_1675


In [5]:
object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

print(f"Number of images: {len(image_db)}")
print(f"Number of objects: {len(object_db)}")

print("\nFirst image keys:")
print(list(image_db.keys())[:10])

print("\nFirst object ids:")
print(list(object_db.keys())[:10])

Number of images: 48
Number of objects: 1262

First image keys:
['alm1pea1', 'alm1pea2', 'alm1pea3', 'alm1pea4', 'alm2pea1', 'alm2pea2', 'alm2pea3', 'alm2pea4', 'alm3pea1', 'alm3pea2']

First object ids:
['alm1pea1_obj001', 'alm1pea1_obj002', 'alm1pea1_obj003', 'alm1pea1_obj004', 'alm1pea1_obj005', 'alm1pea1_obj006', 'alm1pea1_obj007', 'alm1pea1_obj008', 'alm1pea1_obj009', 'alm1pea1_obj010']


In [6]:
if USE_SPECTRAL_RANGE:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=SPECTRAL_MIN_NM,
        max_nm=SPECTRAL_MAX_NM,
    )

    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
    save_parquet(wavelength_selection_df, WAVELENGTH_SELECTION_PATH)

    print("Spectral range selected:")
    display(wavelength_selection_df)
else:
    first_object = next(iter(object_db.values()))
    wavelengths = first_object.get("wavelengths")
    wavelengths = np.asarray(wavelengths) if wavelengths is not None else None

Spectral range selected:


,requested_min_nm,requested_max_nm,actual_min_nm,actual_max_nm,n_original_bands,n_selected_bands,selected_band_indices,selected_wavelengths
0,1225.0,1675.0,1235.720588,1666.132353,63,37,"23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,3...","1235.721,1247.676,1259.632,1271.588,1283.544,1..."


In [7]:
object_rows = []

for object_id, obj in object_db.items():
    object_rows.append({
        "object_id": object_id,
        "source_clean_key": obj.get("source_clean_key"),
        "source_image": obj.get("source_image"),
        "sample_kind": obj.get("sample_kind"),
        "image_nut_type": obj.get("image_nut_type"),
        "object_nut_type": obj.get("object_nut_type"),
        "batch": obj.get("batch"),
        "split": obj.get("split"),
        "position_set": obj.get("position_set"),
        "area_pixels": obj.get("area_pixels"),
        "n_pixels": obj.get("n_pixels"),
        "n_bands": obj.get("n_bands"),
        "is_pure": obj.get("is_pure"),
        "is_mixture": obj.get("is_mixture"),
        "is_position_reference": obj.get("is_position_reference"),
    })

object_meta_df = pd.DataFrame(object_rows)

display(object_meta_df.head())
display(
    object_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
    .sort_values(["sample_kind", "object_nut_type", "batch"], na_position="last")
)

,object_id,source_clean_key,source_image,sample_kind,image_nut_type,object_nut_type,batch,split,position_set,area_pixels,n_pixels,n_bands,is_pure,is_mixture,is_position_reference
0,alm1pea1_obj001,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,projection,NaN,84,84,37,False,True,False
1,alm1pea1_obj002,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,projection,NaN,73,73,37,False,True,False
2,alm1pea1_obj003,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,projection,NaN,91,91,37,False,True,False
3,alm1pea1_obj004,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,projection,NaN,73,73,37,False,True,False
4,alm1pea1_obj005,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,projection,NaN,126,126,37,False,True,False


,sample_kind,object_nut_type,batch,n_objects
0,mixture,unknown,NaN,722
1,position_reference,peanut,1.0,47
2,position_reference,peanut,2.0,47
3,position_reference,peanut,3.0,47
4,position_reference,peanut,4.0,5
5,pure,almond,1.0,52
6,pure,almond,2.0,59
7,pure,almond,3.0,55
8,pure,almond,4.0,48
9,pure,peanut,1.0,46


In [8]:
first_object = next(iter(object_db.values()))
wavelengths = first_object.get("wavelengths")

if wavelengths is not None:
    wavelengths = np.asarray(wavelengths)
    if wavelengths.size == 0:
        wavelengths = None

if wavelengths is None:
    print("No wavelength axis found. Band indices will be used.")
else:
    print("Wavelength axis found.")
    print("n_wavelengths:", len(wavelengths))
    print("first:", wavelengths[:5])
    print("last:", wavelengths[-5:])

Wavelength axis found.
n_wavelengths: 37
first: [1235.72058824 1247.67647059 1259.63235294 1271.58823529 1283.54411765]
last: [1618.30882353 1630.26470588 1642.22058824 1654.17647059 1666.13235294]


## 1. Matrix registry

The modelling pipeline uses a matrix registry to convert `object_db` into numerical matrices.

The main representations are:

- `object_mean`: one row per object using the mean spectrum.
- `object_median`: one row per object using the median spectrum.
- `balanced_pixels`: `m` pixels sampled per object.
- `all_pixels`: every object pixel as one row.

The goal of this section is to check that these matrix builders return coherent shapes, labels and metadata.

In [9]:
methods = available_matrix_methods()

print("Available matrix methods:")
print(methods)

registry_rows = []

for method in methods:
    spec = get_matrix_spec(method)
    registry_rows.append({
        "matrix_method": method,
        "level": spec.level,
        "spectrum_field": spec.spectrum_field,
        "description": spec.description,
        "uses_pixel_sampling": spec.uses_pixel_sampling,
    })

matrix_registry_df = pd.DataFrame(registry_rows)
matrix_registry_df

Available matrix methods:
['all_pixels', 'balanced_pixels', 'object_mean', 'object_median', 'pixel']


,matrix_method,level,spectrum_field,description,uses_pixel_sampling
0,all_pixels,pixel,mean_spectrum,All object pixels as observations.,False
1,balanced_pixels,balanced_pixel,mean_spectrum,m pixels sampled per object.,True
2,object_mean,object,mean_spectrum,One observation per object using the mean spec...,False
3,object_median,object,median_spectrum,One observation per object using the median sp...,False
4,pixel,pixel,mean_spectrum,Alias for all_pixels.,False


In [10]:
def summarize_matrix_output(
    X,
    y,
    meta,
    matrix_method: str,
    filters: dict,
    balanced_pixel_strategy: str | None = None,
):
    """Return a compact summary for a built matrix."""
    X = np.asarray(X)
    y = np.asarray(y)

    meta_df = pd.DataFrame(meta)

    row = {
        "matrix_method": matrix_method,
        "balanced_pixel_strategy": balanced_pixel_strategy,
        "filters": json.dumps(filters),
        "n_observations": int(X.shape[0]),
        "n_features": int(X.shape[1]) if X.ndim == 2 else np.nan,
        "n_labels": int(len(np.unique(y))) if len(y) > 0 else 0,
        "labels": ", ".join(map(str, sorted(pd.Series(y).dropna().unique()))),
        "has_metadata": bool(len(meta_df) == len(y)),
        "n_unique_objects": (
            int(meta_df["object_id"].nunique())
            if "object_id" in meta_df.columns
            else np.nan
        ),
        "n_unique_images": (
            int(meta_df["source_image"].nunique())
            if "source_image" in meta_df.columns
            else np.nan
        ),
        "n_nan_values": int(np.isnan(X).sum()) if np.issubdtype(X.dtype, np.number) else np.nan,
        "nan_rate": float(np.isnan(X).mean()) if np.issubdtype(X.dtype, np.number) else np.nan,
        "global_min": float(np.nanmin(X)) if X.size else np.nan,
        "global_max": float(np.nanmax(X)) if X.size else np.nan,
        "global_mean": float(np.nanmean(X)) if X.size else np.nan,
        "global_std": float(np.nanstd(X)) if X.size else np.nan,
    }

    return row, meta_df

In [11]:
matrix_summary_rows = []
matrix_examples = {}

for matrix_method in MATRIX_METHODS_TO_CHECK:
    if matrix_method == "balanced_pixels":
        for strategy in BALANCED_PIXEL_STRATEGIES:
            print(f"Building matrix={matrix_method}, strategy={strategy}")

            X, y, meta = build_matrix(
                object_db=object_db,
                matrix_method=matrix_method,
                filters=PURE_ALMOND_PEANUT_FILTERS,
                m=M_BALANCED_PIXELS,
                random_state=RANDOM_STATE,
                replace=REPLACE_BALANCED_PIXELS,
                balanced_pixel_strategy=strategy,
            )

            row, meta_df = summarize_matrix_output(
                X=X,
                y=y,
                meta=meta,
                matrix_method=matrix_method,
                filters=PURE_ALMOND_PEANUT_FILTERS,
                balanced_pixel_strategy=strategy,
            )

            matrix_summary_rows.append(row)
            matrix_examples[(matrix_method, strategy)] = {
                "X": X,
                "y": y,
                "meta": meta,
                "meta_df": meta_df,
            }

    else:
        print(f"Building matrix={matrix_method}")

        X, y, meta = build_matrix(
            object_db=object_db,
            matrix_method=matrix_method,
            filters=PURE_ALMOND_PEANUT_FILTERS,
            m=M_BALANCED_PIXELS,
            random_state=RANDOM_STATE,
            replace=REPLACE_BALANCED_PIXELS,
            balanced_pixel_strategy="random",
        )

        row, meta_df = summarize_matrix_output(
            X=X,
            y=y,
            meta=meta,
            matrix_method=matrix_method,
            filters=PURE_ALMOND_PEANUT_FILTERS,
            balanced_pixel_strategy=None,
        )

        matrix_summary_rows.append(row)
        matrix_examples[(matrix_method, None)] = {
            "X": X,
            "y": y,
            "meta": meta,
            "meta_df": meta_df,
        }

matrix_summary_df = pd.DataFrame(matrix_summary_rows)
matrix_summary_df

Building matrix=object_mean
Building matrix=object_median
Building matrix=balanced_pixels, strategy=random
Building matrix=balanced_pixels, strategy=center
Building matrix=all_pixels


,matrix_method,balanced_pixel_strategy,filters,n_observations,n_features,n_labels,labels,has_metadata,n_unique_objects,n_unique_images,n_nan_values,nan_rate,global_min,global_max,global_mean,global_std
0,object_mean,NaN,"{""sample_kind"": [""pure""], ""object_nut_type"": [...",394,37,2,"almond, peanut",True,394,8,0,0.0,0.128382,0.654266,0.310702,0.089586
1,object_median,NaN,"{""sample_kind"": [""pure""], ""object_nut_type"": [...",394,37,2,"almond, peanut",True,394,8,0,0.0,0.115502,0.672581,0.314718,0.094799
2,balanced_pixels,random,"{""sample_kind"": [""pure""], ""object_nut_type"": [...",15440,37,2,"almond, peanut",True,394,8,0,0.0,0.000000,0.991350,0.315097,0.124223
3,balanced_pixels,center,"{""sample_kind"": [""pure""], ""object_nut_type"": [...",15440,37,2,"almond, peanut",True,394,8,0,0.0,0.000000,0.991350,0.339748,0.131488
4,all_pixels,NaN,"{""sample_kind"": [""pure""], ""object_nut_type"": [...",30197,37,2,"almond, peanut",True,394,8,0,0.0,0.000000,0.991350,0.317216,0.126459


In [12]:
save_parquet(matrix_summary_df, MATRIX_SUMMARY_PATH)

print("Saved matrix summary:")
print(MATRIX_SUMMARY_PATH)

Saved matrix summary:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\matrices_preprocessing_1225_1675\matrix_summary.parquet


In [13]:
plot_bar_values(
    x=matrix_summary_df["matrix_method"].astype(str)
      + matrix_summary_df["balanced_pixel_strategy"].fillna("").map(lambda s: f" | {s}" if s else ""),
    y=matrix_summary_df["n_observations"],
    title="Number of observations by matrix representation",
    x_title="matrix method",
    y_title="n observations",
    show=True,
)

In [14]:
for key, item in matrix_examples.items():
    matrix_method, strategy = key
    print("=" * 80)
    print("Matrix:", matrix_method, "| strategy:", strategy)
    print("X shape:", item["X"].shape)
    print("y shape:", item["y"].shape)
    print("metadata shape:", item["meta_df"].shape)
    display(item["meta_df"].head())

Matrix: object_mean | strategy: None
X shape: (394, 37)
y shape: (394,)
metadata shape: (394, 6)


,object_id,label,source_image,batch,area,sample_kind
0,almond1_obj001,almond,almond1,1,54,pure
1,almond1_obj002,almond,almond1,1,95,pure
2,almond1_obj003,almond,almond1,1,52,pure
3,almond1_obj004,almond,almond1,1,98,pure
4,almond1_obj005,almond,almond1,1,84,pure


Matrix: object_median | strategy: None
X shape: (394, 37)
y shape: (394,)
metadata shape: (394, 6)


,object_id,label,source_image,batch,area,sample_kind
0,almond1_obj001,almond,almond1,1,54,pure
1,almond1_obj002,almond,almond1,1,95,pure
2,almond1_obj003,almond,almond1,1,52,pure
3,almond1_obj004,almond,almond1,1,98,pure
4,almond1_obj005,almond,almond1,1,84,pure


Matrix: balanced_pixels | strategy: random
X shape: (15440, 37)
y shape: (15440,)
metadata shape: (15440, 9)


,object_id,label,source_image,batch,area,sample_kind,pixel_index,row,col
0,almond1_obj001,almond,almond1,1,54,pure,41,96,59
1,almond1_obj001,almond,almond1,1,54,pure,2,90,59
2,almond1_obj001,almond,almond1,1,54,pure,48,97,59
3,almond1_obj001,almond,almond1,1,54,pure,4,91,56
4,almond1_obj001,almond,almond1,1,54,pure,3,90,60


Matrix: balanced_pixels | strategy: center
X shape: (15440, 37)
y shape: (15440,)
metadata shape: (15440, 9)


,object_id,label,source_image,batch,area,sample_kind,pixel_index,row,col
0,almond1_obj001,almond,almond1,1,54,pure,27,94,59
1,almond1_obj001,almond,almond1,1,54,pure,26,94,58
2,almond1_obj001,almond,almond1,1,54,pure,20,93,59
3,almond1_obj001,almond,almond1,1,54,pure,34,95,59
4,almond1_obj001,almond,almond1,1,54,pure,28,94,60


Matrix: all_pixels | strategy: None
X shape: (30197, 37)
y shape: (30197,)
metadata shape: (30197, 9)


,object_id,label,source_image,batch,area,sample_kind,pixel_index,row,col
0,almond1_obj001,almond,almond1,1,54,pure,0,90,57
1,almond1_obj001,almond,almond1,1,54,pure,1,90,58
2,almond1_obj001,almond,almond1,1,54,pure,2,90,59
3,almond1_obj001,almond,almond1,1,54,pure,3,90,60
4,almond1_obj001,almond,almond1,1,54,pure,4,91,56


## 2. Balanced pixel strategies

For `balanced_pixels`, two strategies can be compared:

- `random`: randomly sample `m` pixels per object.
- `center`: select the `m` pixels closest to the object centroid.

This is useful because border pixels were often associated with classification errors in previous analyses.

In [15]:
balanced_random = matrix_examples[("balanced_pixels", "random")]["meta_df"].copy()
balanced_center = matrix_examples[("balanced_pixels", "center")]["meta_df"].copy()

balanced_random["strategy"] = "random"
balanced_center["strategy"] = "center"

balanced_meta_df = pd.concat(
    [balanced_random, balanced_center],
    ignore_index=True,
)

display(balanced_meta_df.head())

if "object_id" in balanced_meta_df.columns:
    balanced_counts_df = (
        balanced_meta_df
        .groupby(["strategy", "object_id"], dropna=False)
        .size()
        .reset_index(name="n_selected_pixels")
    )

    display(balanced_counts_df.head())

    strategy_summary_df = (
        balanced_counts_df
        .groupby("strategy")
        .agg(
            n_objects=("object_id", "nunique"),
            selected_pixels_mean=("n_selected_pixels", "mean"),
            selected_pixels_min=("n_selected_pixels", "min"),
            selected_pixels_max=("n_selected_pixels", "max"),
        )
        .reset_index()
    )

    display(strategy_summary_df)
else:
    print("No object_id in metadata; cannot summarize selected pixels by object.")

,object_id,label,source_image,batch,area,sample_kind,pixel_index,row,col,strategy
0,almond1_obj001,almond,almond1,1,54,pure,41,96,59,random
1,almond1_obj001,almond,almond1,1,54,pure,2,90,59,random
2,almond1_obj001,almond,almond1,1,54,pure,48,97,59,random
3,almond1_obj001,almond,almond1,1,54,pure,4,91,56,random
4,almond1_obj001,almond,almond1,1,54,pure,3,90,60,random


,strategy,object_id,n_selected_pixels
0,center,almond1_obj001,40
1,center,almond1_obj002,40
2,center,almond1_obj003,40
3,center,almond1_obj004,40
4,center,almond1_obj005,40


,strategy,n_objects,selected_pixels_mean,selected_pixels_min,selected_pixels_max
0,center,394,39.187817,15,40
1,random,394,39.187817,15,40


In [16]:
balanced_label_summary = (
    balanced_meta_df
    .groupby(["strategy", "label"], dropna=False)
    .size()
    .reset_index(name="n_rows")
    .sort_values(["strategy", "label"])
)

display(balanced_label_summary)

plot_counts_by_group(
    balanced_meta_df,
    group_col="strategy",
    category_col="label",
    title="Balanced pixel rows by strategy and class",
    show=True,
)

,strategy,label,n_rows
0,center,almond,8414
1,center,peanut,7026
2,random,almond,8414
3,random,peanut,7026


## 3. Raw spectra before preprocessing

Before applying preprocessing, we check raw mean spectra from pure almond and pure peanut objects.

In [17]:
X_object_mean = matrix_examples[("object_mean", None)]["X"]
y_object_mean = matrix_examples[("object_mean", None)]["y"]
meta_object_mean = matrix_examples[("object_mean", None)]["meta_df"]

# Optional sampling for readability
if X_object_mean.shape[0] > MAX_SPECTRA_TO_PLOT:
    sampled_indices = []
    temp_df = meta_object_mean.copy()
    temp_df["_row_index"] = np.arange(len(temp_df))

    for label, sub in temp_df.groupby("label", dropna=False):
        sampled = sub.sample(
            n=min(MAX_SPECTRA_TO_PLOT // 2, len(sub)),
            random_state=RANDOM_STATE,
        )
        sampled_indices.extend(sampled["_row_index"].tolist())

    sampled_indices = sorted(sampled_indices)
else:
    sampled_indices = np.arange(X_object_mean.shape[0])

plot_spectra(
    X_object_mean[sampled_indices],
    wavelengths=wavelengths,
    labels=y_object_mean[sampled_indices],
    names=meta_object_mean.iloc[sampled_indices]["object_id"].to_numpy()
        if "object_id" in meta_object_mean.columns
        else None,
    reducer="none",
    title="Raw object mean spectra — pure objects",
    y_title="Reflectance",
    show=True,
)

plot_spectra(
    X_object_mean,
    wavelengths=wavelengths,
    labels=y_object_mean,
    reducer="mean_std",
    title="Raw object mean spectra — mean ± std by class",
    y_title="Reflectance",
    show=True,
)

## 4. Preprocessing configuration registry

The preprocessing registry converts readable names into executable preprocessing chains.

For example:

- `absorbance_snv` → `("absorbance", "snv")`
- `absorbance_sg_smooth` → `("absorbance", "sg_smooth")`
- `absorbance_sg_d1` → `("absorbance", "sg_d1")`

This section verifies that each preprocessing chain can be fitted and applied.

In [18]:
preprocessing_configs = normalize_preprocessing_configs(
    PREPROCESSING_CONFIGS_TO_COMPARE
)

preprocessing_configs_df = pd.DataFrame([
    {
        "preprocessing": name,
        "steps": " + ".join(steps),
        "n_steps": len(steps),
    }
    for name, steps in preprocessing_configs.items()
])

preprocessing_configs_df

,preprocessing,steps,n_steps
0,raw,raw,1
1,absorbance,absorbance,1
2,snv,snv,1
3,msc,msc,1
4,sg_smooth,sg_smooth,1
5,sg_d1,sg_d1,1
6,sg_d2,sg_d2,1
7,snv_sg_smooth,snv + sg_smooth,2
8,snv_sg_d1,snv + sg_d1,2
9,snv_sg_d2,snv + sg_d2,2


In [19]:
preprocessing_results = {}
preprocessing_summary_rows = []

for preprocessing_name, steps in preprocessing_configs.items():
    print(f"Fitting preprocessing: {preprocessing_name} -> {steps}")

    preprocessor = SpectralPreprocessor(
        steps=steps,
        sg_window_length=SG_WINDOW_LENGTH,
        sg_polyorder=SG_POLYORDER,
    )

    X_preprocessed = preprocessor.fit_transform(
        X_object_mean,
        wavelengths=wavelengths,
    )

    preprocessing_results[preprocessing_name] = {
        "steps": steps,
        "preprocessor": preprocessor,
        "X": X_preprocessed,
    }

    preprocessing_summary_rows.append({
        "preprocessing": preprocessing_name,
        "steps": " + ".join(steps),
        "n_observations": int(X_preprocessed.shape[0]),
        "n_features": int(X_preprocessed.shape[1]),
        "global_mean": float(np.nanmean(X_preprocessed)),
        "global_std": float(np.nanstd(X_preprocessed)),
        "global_min": float(np.nanmin(X_preprocessed)),
        "global_max": float(np.nanmax(X_preprocessed)),
        "nan_rate": float(np.mean(~np.isfinite(X_preprocessed))),
        "sg_window_length": SG_WINDOW_LENGTH,
        "sg_polyorder": SG_POLYORDER,
    })

preprocessing_summary_df = pd.DataFrame(preprocessing_summary_rows)

preprocessing_summary_df

Fitting preprocessing: raw -> ('raw',)
Fitting preprocessing: absorbance -> ('absorbance',)
Fitting preprocessing: snv -> ('snv',)
Fitting preprocessing: msc -> ('msc',)
Fitting preprocessing: sg_smooth -> ('sg_smooth',)
Fitting preprocessing: sg_d1 -> ('sg_d1',)
Fitting preprocessing: sg_d2 -> ('sg_d2',)
Fitting preprocessing: snv_sg_smooth -> ('snv', 'sg_smooth')
Fitting preprocessing: snv_sg_d1 -> ('snv', 'sg_d1')
Fitting preprocessing: snv_sg_d2 -> ('snv', 'sg_d2')
Fitting preprocessing: absorbance_snv -> ('absorbance', 'snv')
Fitting preprocessing: absorbance_sg_smooth -> ('absorbance', 'sg_smooth')
Fitting preprocessing: absorbance_sg_d1 -> ('absorbance', 'sg_d1')
Fitting preprocessing: absorbance_sg_d2 -> ('absorbance', 'sg_d2')
Fitting preprocessing: absorbance_snv_sg_smooth -> ('absorbance', 'snv', 'sg_smooth')
Fitting preprocessing: absorbance_snv_sg_d1 -> ('absorbance', 'snv', 'sg_d1')
Fitting preprocessing: absorbance_snv_sg_d2 -> ('absorbance', 'snv', 'sg_d2')


,preprocessing,steps,n_observations,n_features,global_mean,global_std,global_min,global_max,nan_rate,sg_window_length,sg_polyorder
0,raw,raw,394,37,3.107020e-01,0.089586,0.128382,0.654266,0.0,11,2
1,absorbance,absorbance,394,37,5.254638e-01,0.124619,0.184245,0.891497,0.0,11,2
2,snv,snv,394,37,1.827778e-17,0.986394,-1.140261,1.611174,0.0,11,2
3,msc,msc,394,37,3.107020e-01,0.071490,0.228007,0.427923,0.0,11,2
4,sg_smooth,sg_smooth,394,37,3.107785e-01,0.089360,0.129112,0.657746,0.0,11,2
5,sg_d1,sg_d1,394,37,-2.717773e-04,0.000714,-0.002386,0.001769,0.0,11,2
6,sg_d2,sg_d2,394,37,-6.427248e-06,0.000017,-0.000061,0.000034,0.0,11,2
7,snv_sg_smooth,snv + sg_smooth,394,37,1.067931e-03,0.982468,-1.068423,1.639626,0.0,11,2
8,snv_sg_d1,snv + sg_d1,394,37,-3.761064e-03,0.009730,-0.024508,0.021124,0.0,11,2
9,snv_sg_d2,snv + sg_d2,394,37,-8.927741e-05,0.000239,-0.000802,0.000413,0.0,11,2


In [20]:
save_parquet(preprocessing_summary_df, PREPROCESSING_SUMMARY_PATH)
print("Saved preprocessing summary:")
print(PREPROCESSING_SUMMARY_PATH)

Saved preprocessing summary:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\matrices_preprocessing_1225_1675\preprocessing_summary.parquet


In [21]:
for preprocessing_name, result in preprocessing_results.items():
    Xp = result["X"]

    plot_spectra(
        Xp,
        wavelengths=wavelengths,
        labels=y_object_mean,
        reducer="mean_std",
        title=f"{preprocessing_name} — mean ± std by class",
        y_title="Preprocessed value",
        show=True,
    )

In [22]:
SELECTED_PREPROCESSING_FOR_DETAIL = [
    "raw",
    "snv",
    "snv_sg_smooth",
    "snv_sg_d1",
    "snv_sg_d2",
    "absorbance",
    "absorbance_snv",
    "absorbance_sg_smooth",
    "absorbance_sg_d1",
    "absorbance_sg_d2",
    "absorbance_snv_sg_smooth",
    "absorbance_snv_sg_d1",
    "absorbance_snv_sg_d2",
]

available_selected = [
    name for name in SELECTED_PREPROCESSING_FOR_DETAIL
    if name in preprocessing_results
]

print("Available selected preprocessing:")
print(available_selected)

Available selected preprocessing:
['raw', 'snv', 'snv_sg_smooth', 'snv_sg_d1', 'snv_sg_d2', 'absorbance', 'absorbance_snv', 'absorbance_sg_smooth', 'absorbance_sg_d1', 'absorbance_sg_d2', 'absorbance_snv_sg_smooth', 'absorbance_snv_sg_d1', 'absorbance_snv_sg_d2']


In [23]:
for preprocessing_name in available_selected:
    Xp = preprocessing_results[preprocessing_name]["X"]

    plot_spectra(
        Xp[sampled_indices],
        wavelengths=wavelengths,
        labels=y_object_mean[sampled_indices],
        names=meta_object_mean.iloc[sampled_indices]["object_id"].to_numpy()
            if "object_id" in meta_object_mean.columns
            else None,
        reducer="none",
        title=f"{preprocessing_name} — sampled object spectra",
        y_title="Preprocessed value",
        show=True,
    )

## 5. Batch effects after preprocessing

We compare spectra by batch for the main candidate preprocessing methods.

This is especially important because batch 3 was previously observed to be noisy, especially for almonds.

In [24]:
def plot_preprocessing_by_batch(
    preprocessing_name: str,
    target_label: str,
    sample_kind: str = "pure",
):
    """Plot one preprocessing by batch for one target class."""
    if preprocessing_name not in preprocessing_results:
        raise KeyError(f"Unknown preprocessing: {preprocessing_name}")

    Xp = preprocessing_results[preprocessing_name]["X"]

    df = meta_object_mean.copy()
    df["_row_index"] = np.arange(len(df))

    mask = (
        df["label"].astype(str).eq(str(target_label))
        if "label" in df.columns
        else pd.Series(y_object_mean.astype(str)).eq(str(target_label))
    )

    if "sample_kind" in df.columns:
        mask = mask & df["sample_kind"].astype(str).eq(sample_kind)

    sub = df[mask].copy()

    if len(sub) == 0:
        print(f"No object found for label={target_label}, sample_kind={sample_kind}")
        return None

    idx = sub["_row_index"].to_numpy()
    batch_labels = np.asarray([f"batch {b}" for b in sub["batch"].to_numpy()])

    return plot_spectra(
        Xp[idx],
        wavelengths=wavelengths,
        labels=batch_labels,
        reducer="mean_std",
        title=f"{preprocessing_name} — {target_label} spectra by batch",
        y_title="Preprocessed value",
        show=True,
    )

In [25]:
CANDIDATE_BATCH_PREPROCESSINGS = [
    "raw",
    "snv",
    "sg_smooth",
    "sg_d1",
    "sg_d2",
    "snv_sg_smooth",
    "snv_sg_d1",
    "snv_sg_d2",
    "absorbance_snv",
    "absorbance_sg_smooth",
    "absorbance_sg_d1",
    "absorbance_sg_d2",
    "absorbance_snv_sg_d1",
    "absorbance_snv_sg_smooth",
    "absorbance_snv_sg_d2",
]

for preprocessing_name in CANDIDATE_BATCH_PREPROCESSINGS:
    if preprocessing_name not in preprocessing_results:
        continue

    print("=" * 80)
    print("Preprocessing:", preprocessing_name)

    plot_preprocessing_by_batch(
        preprocessing_name=preprocessing_name,
        target_label="almond",
    )

    plot_preprocessing_by_batch(
        preprocessing_name=preprocessing_name,
        target_label="peanut",
    )

Preprocessing: raw


Preprocessing: snv


Preprocessing: sg_smooth


Preprocessing: sg_d1


Preprocessing: sg_d2


...

Preprocessing: absorbance_sg_d1


Preprocessing: absorbance_sg_d2


Preprocessing: absorbance_snv_sg_d1


Preprocessing: absorbance_snv_sg_smooth


Preprocessing: absorbance_snv_sg_d2


## 6. Compare SG smoothing vs SG derivative

A key check is to explicitly compare:

- `sg_smooth`: Savitzky-Golay smoothing with derivative order 0.
- `sg_d1`: Savitzky-Golay 1st derivative.
- `sg_d2`: Savitzky-Golay 2nd derivative.

This confirms that smoothing-only preprocessing is present and can be tested separately from derivative preprocessing.

In [26]:
sg_comparison_names = [
    "sg_smooth",
    "sg_d1",
    "sg_d2",
    "snv_sg_smooth",
    "snv_sg_d1",
    "snv_sg_d2",
    "absorbance_sg_smooth",
    "absorbance_sg_d1",
    "absorbance_sg_d2",
    "absorbance_snv_sg_smooth",
    "absorbance_snv_sg_d1",
    "absorbance_snv_sg_d2",
]

sg_comparison_df = preprocessing_summary_df[
    preprocessing_summary_df["preprocessing"].isin(sg_comparison_names)
].copy()

sg_comparison_df

,preprocessing,steps,n_observations,n_features,global_mean,global_std,global_min,global_max,nan_rate,sg_window_length,sg_polyorder
4,sg_smooth,sg_smooth,394,37,0.310778,0.089360,0.129112,0.657746,0.0,11,2
5,sg_d1,sg_d1,394,37,-0.000272,0.000714,-0.002386,0.001769,0.0,11,2
6,sg_d2,sg_d2,394,37,-0.000006,0.000017,-0.000061,0.000034,0.0,11,2
7,snv_sg_smooth,snv + sg_smooth,394,37,0.001068,0.982468,-1.068423,1.639626,0.0,11,2
8,snv_sg_d1,snv + sg_d1,394,37,-0.003761,0.009730,-0.024508,0.021124,0.0,11,2
9,snv_sg_d2,snv + sg_d2,394,37,-0.000089,0.000239,-0.000802,0.000413,0.0,11,2
11,absorbance_sg_smooth,absorbance + sg_smooth,394,37,0.525336,0.124303,0.181726,0.888841,0.0,11,2
12,absorbance_sg_d1,absorbance + sg_d1,394,37,0.000391,0.000944,-0.001901,0.003216,0.0,11,2
13,absorbance_sg_d2,absorbance + sg_d2,394,37,0.000009,0.000025,-0.000055,0.000100,0.0,11,2
14,absorbance_snv_sg_smooth,absorbance + snv + sg_smooth,394,37,-0.001315,0.982406,-1.581192,1.148700,0.0,11,2


In [27]:
for pair in [
    ("sg_smooth", "sg_d1"),
    ("sg_smooth", "sg_d2"),
    ("snv_sg_smooth", "snv_sg_d1"),
    ("snv_sg_smooth", "snv_sg_d2"),
    ("absorbance_sg_smooth", "absorbance_sg_d1"),
    ("absorbance_sg_smooth", "absorbance_sg_d2"),
    ("absorbance_snv_sg_smooth", "absorbance_snv_sg_d1"),
    ("absorbance_snv_sg_smooth", "absorbance_snv_sg_d2"),
]:
    smooth_name, d1_name = pair

    if smooth_name not in preprocessing_results or d1_name not in preprocessing_results:
        print(f"Skipping pair {pair}: one preprocessing is missing.")
        continue

    print("=" * 80)
    print("Comparing:", smooth_name, "vs", d1_name)

    plot_spectra(
        preprocessing_results[smooth_name]["X"],
        wavelengths=wavelengths,
        labels=y_object_mean,
        reducer="mean_std",
        title=f"{smooth_name} — mean ± std by class",
        y_title="Preprocessed value",
        show=True,
    )

    plot_spectra(
        preprocessing_results[d1_name]["X"],
        wavelengths=wavelengths,
        labels=y_object_mean,
        reducer="mean_std",
        title=f"{d1_name} — mean ± std by class",
        y_title="Preprocessed value",
        show=True,
    )

Comparing: sg_smooth vs sg_d1


Comparing: sg_smooth vs sg_d2


Comparing: snv_sg_smooth vs snv_sg_d1


Comparing: snv_sg_smooth vs snv_sg_d2


Comparing: absorbance_sg_smooth vs absorbance_sg_d1


Comparing: absorbance_sg_smooth vs absorbance_sg_d2


Comparing: absorbance_snv_sg_smooth vs absorbance_snv_sg_d1


Comparing: absorbance_snv_sg_smooth vs absorbance_snv_sg_d2


## 7. Matrix + preprocessing compatibility

We now check that each preprocessing can be applied to each relevant matrix representation.

This is a smoke test for the next notebooks:

- PCA comparison,
- SIMCA selection,
- Optuna / grid search.

In [28]:
compatibility_rows = []

for matrix_key, item in matrix_examples.items():
    matrix_method, strategy = matrix_key
    X_raw = item["X"]

    for preprocessing_name, steps in preprocessing_configs.items():
        try:
            preprocessor = SpectralPreprocessor(
                steps=steps,
                sg_window_length=SG_WINDOW_LENGTH,
                sg_polyorder=SG_POLYORDER,
            )

            Xp = preprocessor.fit_transform(
                X_raw,
                wavelengths=wavelengths,
            )

            status = "ok"
            error = ""
            n_observations, n_features = Xp.shape
            nan_rate = float(np.mean(~np.isfinite(Xp)))

        except Exception as exc:
            status = "error"
            error = repr(exc)
            n_observations = np.nan
            n_features = np.nan
            nan_rate = np.nan

        compatibility_rows.append({
            "matrix_method": matrix_method,
            "balanced_pixel_strategy": strategy,
            "preprocessing": preprocessing_name,
            "steps": " + ".join(steps),
            "status": status,
            "error": error,
            "n_observations": n_observations,
            "n_features": n_features,
            "nan_rate": nan_rate,
        })

compatibility_df = pd.DataFrame(compatibility_rows)

display(compatibility_df)

errors_df = compatibility_df[compatibility_df["status"].ne("ok")].copy()

if errors_df.empty:
    print("All matrix/preprocessing combinations passed.")
else:
    print("[WARNING] Some combinations failed.")
    display(errors_df)

,matrix_method,balanced_pixel_strategy,preprocessing,steps,status,error,n_observations,n_features,nan_rate
0,object_mean,NaN,raw,raw,ok,,394,37,0.0
1,object_mean,NaN,absorbance,absorbance,ok,,394,37,0.0
2,object_mean,NaN,snv,snv,ok,,394,37,0.0
3,object_mean,NaN,msc,msc,ok,,394,37,0.0
4,object_mean,NaN,sg_smooth,sg_smooth,ok,,394,37,0.0
5,object_mean,NaN,sg_d1,sg_d1,ok,,394,37,0.0
6,object_mean,NaN,sg_d2,sg_d2,ok,,394,37,0.0
7,object_mean,NaN,snv_sg_smooth,snv + sg_smooth,ok,,394,37,0.0
8,object_mean,NaN,snv_sg_d1,snv + sg_d1,ok,,394,37,0.0
9,object_mean,NaN,snv_sg_d2,snv + sg_d2,ok,,394,37,0.0


All matrix/preprocessing combinations passed.


In [29]:
compatibility_csv = RESULTS_DIR / "matrix_preprocessing_compatibility.csv"
compatibility_df.to_csv(compatibility_csv, index=False)

print("Saved compatibility summary:")
print(compatibility_csv)

Saved compatibility summary:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\matrices_preprocessing_1225_1675\matrix_preprocessing_compatibility.csv


## 8. Recommended preprocessing sets for next notebooks

For PCA, we can keep a wider set of preprocessing methods.

For SIMCA, we usually keep a smaller and more focused set, especially:

- `snv`
- `absorbance`
- `absorbance_snv`
- `absorbance_sg_smooth`
- `absorbance_sg_d1`
- `absorbance_sg_d2`
- `snv_sg_smooth`
- `snv_sg_d1`
- `snv_sg_d2`
- `absorbance_snv_sg_smooth`
- `absorbance_snv_sg_d1`
- `absorbance_snv_sg_d2`

The final choice will be made in the PCA and SIMCA notebooks.

In [30]:
default_configs_df = pd.DataFrame([
    {
        "set": "default",
        "preprocessing": name,
        "steps": " + ".join(steps),
    }
    for name, steps in normalize_preprocessing_configs(DEFAULT_PREPROCESSING_CONFIGS).items()
])

simca_configs_df = pd.DataFrame([
    {
        "set": "simca_search",
        "preprocessing": name,
        "steps": " + ".join(steps),
    }
    for name, steps in normalize_preprocessing_configs(SIMCA_SEARCH_PREPROCESSING_CONFIGS).items()
])

preprocessing_sets_df = pd.concat(
    [default_configs_df, simca_configs_df],
    ignore_index=True,
)

preprocessing_sets_df

,set,preprocessing,steps
0,default,raw,raw
1,default,absorbance,absorbance
2,default,snv,snv
3,default,msc,msc
4,default,sg_smooth,sg_smooth
5,default,sg_d1,sg_d1
6,default,sg_d2,sg_d2
7,default,absorbance_snv,absorbance + snv
8,default,absorbance_msc,absorbance + msc
9,default,absorbance_sg_smooth,absorbance + sg_smooth


In [31]:
notebook_config = {
    "db_h5_path": str(DB_H5_PATH),
    "results_dir": str(RESULTS_DIR),
    "matrix_methods_to_check": MATRIX_METHODS_TO_CHECK,
    "balanced_pixel_strategies": BALANCED_PIXEL_STRATEGIES,
    "m_balanced_pixels": int(M_BALANCED_PIXELS),
    "replace_balanced_pixels": bool(REPLACE_BALANCED_PIXELS),
    "random_state": int(RANDOM_STATE),
    "pure_filters": PURE_FILTERS,
    "pure_almond_peanut_filters": PURE_ALMOND_PEANUT_FILTERS,
    "sg_window_length": int(SG_WINDOW_LENGTH),
    "sg_polyorder": int(SG_POLYORDER),
    "preprocessing_configs_to_compare": {
        name: list(steps)
        for name, steps in preprocessing_configs.items()
    },
    "matrix_summary": str(MATRIX_SUMMARY_PATH),
    "preprocessing_summary": str(PREPROCESSING_SUMMARY_PATH),
    "compatibility_csv": str(compatibility_csv),
}

with open(MATRIX_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(notebook_config, f, indent=2)

print("Saved notebook config:")
print(MATRIX_CONFIG_JSON)

Saved notebook config:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\matrices_preprocessing_1225_1675\matrices_preprocessing_config.json


In [32]:
print("02_matrices_preprocessing.ipynb completed.")
print()
print("Main outputs:")
print(" -", MATRIX_SUMMARY_PATH)
print(" -", PREPROCESSING_SUMMARY_PATH)
print(" -", compatibility_csv)
print(" -", MATRIX_CONFIG_JSON)
print()
print("Recommended next notebook:")
print("03_pca_exploration_selection.ipynb")
print()
print("Main checks:")
print(f" - Matrix combinations checked: {len(matrix_summary_df)}")
print(f" - Preprocessing methods checked: {len(preprocessing_summary_df)}")
print(f" - Compatibility errors: {len(errors_df)}")

02_matrices_preprocessing.ipynb completed.

Main outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\matrices_preprocessing_1225_1675\matrix_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\matrices_preprocessing_1225_1675\preprocessing_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\matrices_preprocessing_1225_1675\matrix_preprocessing_compatibility.csv
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\matrices_preprocessing_1225_1675\matrices_preprocessing_config.json

Recommended next notebook:
03_pca_exploration_selection.ipynb

Main checks:
 - Matrix combinations checked: 5
 - Preprocessing methods checked: 17
 - Compatibility errors: 0
